# Data Coverage Analysis

This notebook evaluates the completeness of the temperature dataset at annual, monthly, and seasonal levels.

For each period, data coverage is calculated as the percentage of available days relative to the expected number of days. Periods with less than 70% coverage are excluded from the corresponding analysis dataset.

Minimum temperature (`tmin`) coverage is also assessed separately because `tmin` contains additional missing values. A period can therefore meet the general coverage threshold while having lower `tmin` coverage.

## Imports and loading data

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


DATA_FILE = Path("../data/processed/sofia_daily_temperature.csv")

df = pd.read_csv(DATA_FILE)

df["time"] = pd.to_datetime(df["time"])

## Annual Data Coverage

The annual analysis considers complete years from 1952 to 2024. The year 2025 is excluded from annual coverage analysis because the dataset ends on August 22, 2025.

In [ ]:
annual_df = df[df["year"] < 2025].copy()

days_by_year = (annual_df.groupby("year")["time"].nunique())

expected_days_by_year = pd.Series({
    year:
    366 if pd.Timestamp(year, 1, 1).is_leap_year else 365
    for year in days_by_year.index
})

annual_coverage = pd.DataFrame({
    "available_days": days_by_year,
    "expected_days": expected_days_by_year
})

annual_coverage["coverage_percentage"] = (annual_coverage["available_days"] /
                                          annual_coverage["expected_days"] *
                                          100).round(2)

annual_coverage

In [ ]:
annual_coverage["coverage_percentage"].plot(
    figsize=(16, 6)
)

plt.xlabel("Year")
plt.ylabel("Coverage (%)")
plt.title("Annual Data Coverage")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### Coverage Thresholds

Different coverage thresholds are compared to determine how many years would remain available for analysis.

The 70% threshold is used for the final analysis in order to balance data completeness with the amount of historical data retained.

In [ ]:
thresholds = [50, 60, 70, 80, 90, 95]

for threshold in thresholds:
    years = (
        annual_coverage["coverage_percentage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{years} years retained"
    )

### Applying the 70% Coverage Threshold

In [ ]:
MIN_COVERAGE = 70
annual_coverage_filtered = annual_coverage[
    annual_coverage["coverage_percentage"] >= MIN_COVERAGE
].copy()

### Minimum Temperature Coverage

`tcoverage` represents the percentage of available days for which a minimum temperature value (`tmin`) is present.

In [ ]:
annual_coverage_filtered["tcoverage"] = (
    df.groupby("year")["tmin"]
    .count()
    .div(
        df.groupby("year")["tmin"].size()
    )
    .mul(100)
    .round(2)
)

In [ ]:
annual_coverage_filtered["tcoverage"].plot(
    figsize=(16, 6)
)

plt.xlabel("Year")
plt.ylabel("Tmin coverage (%)")
plt.title("Annual Minimum Temperature Data Coverage")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### Minimum Temperature Coverage Thresholds

The same coverage thresholds are examined for `tmin`.

Low `tmin` coverage does not remove the year from the overall dataset.

In [ ]:
thresholds = [50, 60, 70, 80, 90, 95]

for threshold in thresholds:
    years = (
        annual_coverage_filtered["tcoverage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{years} years retained"
    )

## Monthly Data Coverage

The same coverage analysis process is applied at the year-month level.

In [ ]:
monthly_coverage = (
    df.groupby(["year", "month"])["time"]
    .nunique()
    .rename("available_days")
    .to_frame()
)

monthly_coverage["expected_days"] = [
    (
        min(
            pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(1),
            df["time"].max()
        )
        - max(
            pd.Timestamp(year, month, 1),
            df["time"].min()
        )
    ).days + 1
    for year, month in monthly_coverage.index
]

monthly_coverage["coverage_percentage"] = (
    monthly_coverage["available_days"]
    / monthly_coverage["expected_days"]
    * 100
).round(2)

monthly_coverage

In [ ]:
monthly_thresholds = [50, 60, 70, 80, 90, 95]

for threshold in monthly_thresholds:
    retained_months = (
        monthly_coverage["coverage_percentage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{retained_months} month-periods retained"
    )

In [ ]:
MIN_COVERAGE = 70
monthly_coverage_filtered = monthly_coverage[
    monthly_coverage["coverage_percentage"] >= MIN_COVERAGE
].copy()

In [ ]:
monthly_coverage_filtered["tcoverage"] = (
    df.groupby(["year", "month"])["tmin"]
    .count()
    .div(
        df.groupby(["year", "month"])["tmin"].size()
    )
    .mul(100)
    .round(2)
)

In [ ]:
tmin_monthly_thresholds = [50, 60, 70, 80, 90, 95]

for threshold in tmin_monthly_thresholds:
    retained_months = (
        monthly_coverage_filtered["tcoverage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{retained_months} month-periods retained"
    )

## Seasonal Data Coverage

The same coverage analysis process is applied at the season-year level.

### Season-Year Definition

Winter is assigned to the year in which January and February occur. For example, December 1994, January 1995, and February 1995 are treated as Winter 1995.

In [ ]:
df["season_year"] = df["year"]

df.loc[df["month"] == 12, "season_year"] += 1

seasonal_coverage = (
    df.groupby(["season_year", "season"])["time"]
    .nunique()
    .rename("available_days")
    .to_frame()
)

expected_days = []

for (season_year, season) in seasonal_coverage.index:

    if season == "Winter":
        days = 62 + pd.Timestamp(season_year, 2, 1).days_in_month
    elif season == "Spring":
        days = 92
    elif season == "Summer":
        days = 92
    else:
        days = 91

    if season_year == 1952 and season == "Winter":
        days = 31 + 29

    if season_year == 2025 and season == "Summer":
        days = 30 + 31 + 22

    expected_days.append(days)

seasonal_coverage["expected_days"] = expected_days

seasonal_coverage["coverage_percentage"] = (
    seasonal_coverage["available_days"]
    / seasonal_coverage["expected_days"]
    * 100
).round(2)

seasonal_coverage

In [ ]:
seasonal_thresholds = [50, 60, 70, 80, 90, 95]

for threshold in seasonal_thresholds:
    retained_seasons = (
        seasonal_coverage["coverage_percentage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{retained_seasons} season-periods retained"
    )

In [ ]:
MIN_COVERAGE = 70
seasonal_coverage_filtered = seasonal_coverage[
    seasonal_coverage["coverage_percentage"] >= MIN_COVERAGE
].copy()

In [ ]:
seasonal_coverage_filtered["tcoverage"] = (
    df.groupby(["season_year", "season"])["tmin"]
    .count()
    .div(
        df.groupby(["season_year", "season"])["tmin"].size()
    )
    .mul(100)
    .round(2)
)

In [ ]:
tmin_seasonal_thresholds = [50, 60, 70, 80, 90, 95]

for threshold in tmin_seasonal_thresholds:
    retained_seasons = (
        seasonal_coverage_filtered["tcoverage"] >= threshold
    ).sum()

    print(
        f"{threshold}% threshold: "
        f"{retained_seasons} season-periods retained"
    )

## Saving the Coverage Datasets

The filtered annual, monthly, and seasonal datasets are saved separately in the processed data directory.

The original cleaned daily dataset remains unchanged.

In [ ]:
processed_dir = Path("../data/processed")

annual_coverage_filtered.index.name = "year"

annual_coverage_filtered[
    ["coverage_percentage", "tcoverage"]
].to_csv(
    processed_dir / "annual_coverage.csv",
    index=True
)

monthly_coverage_filtered[
    ["coverage_percentage", "tcoverage"]
].to_csv(
    processed_dir / "monthly_coverage.csv",
    index=True
)

seasonal_coverage_filtered[
    ["coverage_percentage", "tcoverage"]
].to_csv(
    processed_dir / "seasonal_coverage.csv",
    index=True
)

print("Coverage datasets saved successfully.")